# Mini-Project Sprint: Multi-Agent Travel Planner

**Prompt Engineering & Autonomous Reasoning**

This practical builds a complete manager-worker travel-planning system.

Earlier in this module, you practised prompt structure, ReAct-style tool use, and Reflexion-style critique. This sprint turns those techniques into one inspectable project:

- Prompt contracts turn vague tasks into structured inputs and outputs.
- ReAct gives each worker an inspectable reason → action → observation → decision trace.
- Reflexion lets the system critique and improve the first draft.
- Local tools simulate external travel APIs so the session can focus on orchestration.


## Practical outcome

You will run and inspect a travel-planner agent team:

1. A manager extracts requirements from a travel request.
2. Flight, hotel, and activity workers use local tools.
3. The manager synthesises a first itinerary.
4. A critic checks the itinerary.
5. The manager returns a refined final plan.

In [ ]:
# TRAINER NOTE: Run this first to confirm the notebook is being executed from the project root.
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
print("Project root:", PROJECT_ROOT)

if not (PROJECT_ROOT / "main.py").exists():
    raise RuntimeError("Open this notebook from the mini_project_sprint project folder.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

## Setup and runtime mode

The project supports two modes:

- `offline`: deterministic classroom demo with no API key required.
- `live`: real OpenAI calls for requirement extraction, worker ranking, synthesis, critique, and final refinement.

Start in `offline` mode first. Switch to `live` only after the class understands the architecture.

In [ ]:
from utils import load_project_env, pretty_json

settings = load_project_env()
print(pretty_json(settings))

# COST NOTE: Offline mode costs ₹0. Live mode uses several compact gpt-4o-mini calls.

## Inspect the local travel tools

The tools simulate travel APIs. They are intentionally deterministic so the source of each fact is visible.

In [ ]:
from tools.travel_tools import load_travel_data

travel_data = load_travel_data()
print("Flights:", len(travel_data["flights"]))
print("Hotels:", len(travel_data["hotels"]))
print("Activities:", len(travel_data["activities"]))

print("\nSample flight:")
print(pretty_json(travel_data["flights"][0]))

## Step 1 — Start with a realistic user request

The request includes hard constraints and preferences. The manager must separate them before delegating work.

In [ ]:
user_request = (
    "Plan a 4-day trip from Mumbai to Singapore for two people. Keep it mid-budget, "
    "avoid red-eye flights, prefer food, city views, and cultural sites. "
    "Keep the total hotel budget under ₹45,000."
)

print(user_request)

## Step 2 — Create the manager

The manager owns orchestration. It does not directly search flights or hotels; it delegates to workers.

In [ ]:
from utils import LLMClient
from agents.manager import TravelPlannerManager

llm = LLMClient(
    demo_mode=settings["demo_mode"],
    model_name=settings["model_name"],
    temperature=float(settings["temperature"]),
)

manager = TravelPlannerManager(llm=llm, travel_data=travel_data)
print("Manager ready in", settings["demo_mode"], "mode")

## Step 3 — Manager extracts requirements

This is the first prompt contract. Free text becomes a structured object that every worker can consume.

In [ ]:
requirements = manager.parse_request(user_request)
print(pretty_json(requirements))

### What to notice

The manager should extract:

- route
- trip duration
- traveller count
- hotel budget
- hard constraints such as avoiding red-eye flights
- preferences such as food, views, and culture

If the extraction is weak, every downstream worker becomes less reliable.

## Step 4 — Run the workers individually

Each worker uses the same pattern:

`Thought → Action → Observation → Decision`

This is ReAct as an auditable worker trace, not an uncontrolled chain-of-thought dump.

In [ ]:
flight_report = manager.flight_worker.run(requirements)
print(pretty_json(flight_report))

In [ ]:
hotel_report = manager.hotel_worker.run(requirements)
print(pretty_json(hotel_report))

In [ ]:
activity_report = manager.activity_worker.run(requirements)
print(pretty_json(activity_report))

## Step 5 — Combine worker outputs

The manager now has enough grounded information to build a first itinerary.

In [ ]:
worker_outputs = {
    "flight": flight_report,
    "hotel": hotel_report,
    "activities": activity_report,
}

draft_itinerary = manager.create_draft_itinerary(requirements, worker_outputs)
print(pretty_json(draft_itinerary))

### What to notice

The draft should be grounded in worker outputs:

- selected flight comes from local flight data
- selected hotel comes from local hotel data
- activities come from local activity data
- budget is calculated, not guessed

This separation is what makes the system debuggable.

## Step 6 — Reflexion critique

The critic checks the first draft for constraint violations, missing preferences, pacing issues, and unsupported assumptions.

In [ ]:
critique = manager.critic.evaluate(requirements, draft_itinerary, worker_outputs)
print(pretty_json(critique))

## Step 7 — Final refinement

The final manager applies the critic's feedback and exposes what changed.

In [ ]:
final_output = manager.refine_itinerary(requirements, draft_itinerary, critique, worker_outputs)
print(pretty_json(final_output))

## Step 8 — Run the full pipeline end-to-end

Now run the same process as one complete manager-worker-reflexion pipeline.

In [ ]:
full_result = manager.run(user_request)
print(pretty_json(full_result["final_output"]))

## Debug view

When something goes wrong, inspect the state in this order:

1. Did requirement extraction capture the right constraints?
2. Did each worker return grounded options?
3. Did synthesis preserve the worker recommendations?
4. Did the critic find actionable issues?
5. Did the final manager actually revise the answer?

In [ ]:
print("Requirement extraction:")
print(pretty_json(full_result["requirements"]))

print("\nCritique:")
print(pretty_json(full_result["critique"]))

## Exercise 1 — Change the request

Try a different trip request using the same local data.

Hint: Mumbai → Singapore and Mumbai → Dubai are represented in the sample data.

In [ ]:
# Try this:
# custom_request = "Plan a 3-day Dubai trip from Mumbai for two people. Prefer culture, food, and metro access."
# custom_result = manager.run(custom_request)
# print(pretty_json(custom_result["final_output"]))

## Exercise 2 — Add a worker improvement

Open `tools/travel_tools.py` and change one ranking rule.

Ideas:

- penalise early-morning flights
- prioritise metro access more strongly
- add a stronger penalty for hotels over budget

Run the same request before and after the change.

## Exercise 3 — Add a new critic rule

Open `agents/critic.py` and add one check.

Ideas:

- flag days with more than two major activities
- flag activities above a cost threshold
- require at least one meal-oriented activity for food-focused trips

## Closing summary

This sprint turns prompting techniques into a working agentic system:

- prompt contracts define reliable communication
- ReAct makes worker tool use inspectable
- manager-worker orchestration prevents one giant prompt from doing everything
- Reflexion improves the first draft before the final answer
- local tools keep the classroom build reliable while preserving the production architecture